# 01 · Dữ liệu + baseline bigram

Chạy trên Kaggle hoặc Colab, **CPU là đủ**, nhớ bật Internet (Kaggle: Settings → Internet on).
Kaggle: Settings → Accelerator → None (CPU). Colab: runtime CPU mặc định là được.
Input: Kaggle dùng dataset `vi-diacritics-code` (code của repo này); Colab tự clone repo từ GitHub.

Output của notebook này (`data/splits`, `baseline.json`, các file eval) được notebook 02, 03, 04 dùng lại.
Trên Colab output nằm trong Google Drive (`MyDrive/ai-portfolio/vi-diacritics-transformer`).

In [ ]:
import glob, json, os, sys, time

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_diacritics/__init__.py", recursive=True)[0].rsplit("/vi_diacritics", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-diacritics-transformer" if IN_COLAB else os.path.abspath("vi-diacritics-transformer")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-diacritics-transformer {REPO}
    else:  # đã clone ở lần chạy trước -> lấy bản code mới nhất
        !git -C {REPO} pull -q
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vi-diacritics-transformer" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE
!pip install -q "datasets>=2.19" tokenizers

## Wikipedia tiếng Việt → câu sạch
Stream thẳng từ HF (`wikimedia/wikipedia`, bản `20231101.vi`, ~730MB parquet), không cần tải hết về.
Lọc câu quá ngắn/dài, câu nhiều ký tự lạ, câu có < 30% âm tiết mang dấu (tiếng Anh, tên riêng...).
Chia train/val/test theo hash của câu để câu trùng không rơi vào 2 tập khác nhau.

In [ ]:
from vi_diacritics.data import build_splits, iter_wikipedia

# đã có từ lần chạy trước (lưu trên Drive) thì bỏ qua, đỡ mất ~10 phút
if os.path.exists("data/splits/test.txt") and os.path.getsize("data/splits/train.txt") > 100_000_000:
    print("data/splits đã có, bỏ qua")
else:
    t0 = time.time()
    counts = build_splits(iter_wikipedia(), "data/splits", max_train=2_000_000)
    print(counts, f"{(time.time() - t0) / 60:.1f} phút")

In [ ]:
!head -n 5 data/splits/train.txt
!wc -l data/splits/*.txt

## Baseline: bigram âm tiết
Với mỗi âm tiết không dấu, chọn dạng có dấu hay gặp nhất sau từ đứng trước. Không "học" gì, chỉ đếm.

In [ ]:
from vi_diacritics.baseline import NgramBaseline
from vi_diacritics.dataset import read_lines
from vi_diacritics.labels import strip
from vi_diacritics.metrics import confusions, evaluate

if os.path.exists("baseline.json"):
    bl = NgramBaseline.load("baseline.json")
else:
    train = read_lines("data/splits/train.txt")
    t0 = time.time()
    bl = NgramBaseline().fit(train)
    bl.prune(2)
    bl.save("baseline.json")
    print(f"fit {(time.time() - t0) / 60:.1f} phút")
print(f"{len(bl.uni):,} âm tiết, {len(bl.bi):,} bigram")

test = read_lines("data/splits/test.txt", 5000)
preds = [bl.restore(strip(s)) for s in test]
print(evaluate(preds, test))
confusions(preds, test, top=15)

## Bộ test ngoài: `nrl-ai/vn-diacritic-eval`
1.227 câu, 4 thể loại (kinh doanh, trang trọng, hội thoại, văn học). Dùng để xem model train trên
Wikipedia (văn viết) yếu ở thể loại nào.

In [ ]:
from datasets import get_dataset_config_names, load_dataset
from vi_diacritics.data import accent_ratio, normalize_text

# mỗi thể loại là một config riêng: business_55, conversational_300, formal_72, literary_800
configs = get_dataset_config_names("nrl-ai/vn-diacritic-eval")
print(configs)
os.makedirs("data/eval", exist_ok=True)
all_lines = []
for cfg in configs:
    ev = load_dataset("nrl-ai/vn-diacritic-eval", cfg)
    ds = ev[list(ev.keys())[0]]
    # cột chứa câu có dấu = cột text có tỉ lệ âm tiết mang dấu cao nhất
    text_cols = [c for c in ds.column_names if isinstance(ds[0][c], str)]
    sample = ds.select(range(min(50, len(ds))))
    col = max(text_cols, key=lambda c: sum(accent_ratio(r[c]) for r in sample))
    lines = [normalize_text(r[col]) for r in ds if r[col].strip()]
    name = cfg.rsplit("_", 1)[0]
    with open(f"data/eval/nrl_{name}.txt", "w", encoding="utf-8") as f:
        f.writelines(ln + "\n" for ln in lines)
    all_lines += lines
    print(cfg, "cột", col, len(lines), "câu, vd:", lines[0][:80])
with open("data/eval/nrl_all.txt", "w", encoding="utf-8") as f:
    f.writelines(ln + "\n" for ln in all_lines)
!wc -l data/eval/*.txt

In [ ]:
for p in sorted(glob.glob("data/eval/nrl_*.txt")):
    gold = read_lines(p)
    print(os.path.basename(p), evaluate([bl.restore(strip(s)) for s in gold], gold))